In [1]:
import os
import sys
import types
import importlib.util
import importlib.machinery

os.environ["TORCH_CUDA_ARCH_LIST"] = "9.0"
os.environ["UNSLOTH_FORCE_COLAB"] = "1"

# Build a proper mock with a real ModuleSpec (not None)
mock_tv = types.ModuleType("torchvision")
mock_tv.__spec__ = importlib.machinery.ModuleSpec("torchvision", loader=None)
mock_tv.__path__ = []
mock_tv.__package__ = "torchvision"
sys.modules["torchvision"] = mock_tv

for sub in ["torchvision.ops", "torchvision.transforms", "torchvision.models", 
            "torchvision.io", "torchvision.utils", "torchvision.datasets"]:
    mod = types.ModuleType(sub)
    mod.__spec__ = importlib.machinery.ModuleSpec(sub, loader=None)
    sys.modules[sub] = mod

import torch
torch.cuda.get_device_capability = lambda x=None: (9, 0)

print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Torch version: {torch.__version__}")
print(f"GPU: {torch.cuda.get_device_name(0)}")

import unsloth
from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments
from datasets import load_dataset

print("All imports successful!")

CUDA available: True
Torch version: 2.7.0.dev20250310+cu124
GPU: NVIDIA GeForce RTX 5060 Ti
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


c:\Users\Admin\miniconda3\envs\unsloth_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Exception: cannot import name 'InterpolationMode' from 'torchvision.transforms' (unknown location)

In [1]:
import os
import sys
import types
import importlib.util
import importlib.machinery

os.environ["TORCH_CUDA_ARCH_LIST"] = "9.0"
os.environ["UNSLOTH_FORCE_COLAB"] = "1"

# ── Build mock helper ───────────────────────────────────────────────────────
def make_mock(name):
    mod = types.ModuleType(name)
    mod.__spec__ = importlib.machinery.ModuleSpec(name, loader=None)
    mod.__path__ = []
    mod.__package__ = name.split(".")[0]
    return mod

# ── InterpolationMode stub ──────────────────────────────────────────────────
class _InterpolationMode:
    NEAREST = NEAREST_EXACT = BOX = BILINEAR = BICUBIC = LANCZOS = HAMMING = 0

# ── v2.functional stub (just needs to be importable) ───────────────────────
tv_v2_functional = make_mock("torchvision.transforms.v2.functional")
tv_v2 = make_mock("torchvision.transforms.v2")
tv_v2.functional = tv_v2_functional

tv_transforms = make_mock("torchvision.transforms")
tv_transforms.InterpolationMode = _InterpolationMode
tv_transforms.v2 = tv_v2

tv_root = make_mock("torchvision")
tv_root.transforms = tv_transforms

# Register everything
for name, mod in [
    ("torchvision",                          tv_root),
    ("torchvision.transforms",               tv_transforms),
    ("torchvision.transforms.v2",            tv_v2),
    ("torchvision.transforms.v2.functional", tv_v2_functional),
    ("torchvision.ops",     make_mock("torchvision.ops")),
    ("torchvision.models",  make_mock("torchvision.models")),
    ("torchvision.io",      make_mock("torchvision.io")),
    ("torchvision.utils",   make_mock("torchvision.utils")),
    ("torchvision.datasets",make_mock("torchvision.datasets")),
]:
    sys.modules[name] = mod

# ── Patch transformers to report torchvision as unavailable ────────────────
import transformers.utils.import_utils as _tv_check
_tv_check._torchvision_available = False

import torch
torch.cuda.get_device_capability = lambda x=None: (9, 0)

print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Torch: {torch.__version__}")
print(f"GPU: {torch.cuda.get_device_name(0)}")

import unsloth
from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments
from datasets import load_dataset

print("All imports successful!")

# ── Load model ──────────────────────────────────────────────────────────────
max_seq_length = 2048
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/meta-llama-3.1-8b-bnb-4bit",
    max_seq_length = max_seq_length,
    load_in_4bit = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)
print("Model + LoRA ready!")

c:\Users\Admin\miniconda3\envs\unsloth_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


CUDA available: True
Torch: 2.7.0.dev20250310+cu124
GPU: NVIDIA GeForce RTX 5060 Ti
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


C:\Users\Admin\AppData\Local\Temp\ipykernel_21696\1821674909.py:59: UserWarning: WARNING: Unsloth should be imported before [transformers] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  import unsloth
W0316 03:46:03.005000 21696 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


🦥 Unsloth Zoo will now patch everything to make training faster!
All imports successful!
==((====))==  Unsloth 2026.3.4: Fast Llama patching. Transformers: 5.2.0.
   \\   /|    NVIDIA GeForce RTX 5060 Ti. Num GPUs = 1. Max memory: 15.928 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.7.0.dev20250310+cu124. CUDA: 12.0. CUDA Toolkit: 12.4. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


RuntimeError: CUDA error: no kernel image is available for execution on the device
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [ ]:
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token 

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input, output in zip(instructions, inputs, outputs):
        text = alpaca_prompt.format(instruction, input, output) + EOS_TOKEN
        texts.append(text)
    return { "text" : texts, }

# Load your cleaned dataset
dataset = load_dataset("json", data_files="therapy_dataset_final.jsonl", split="train")
dataset = dataset.map(formatting_prompts_func, batched = True)

In [ ]:
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 10,
        num_train_epochs = 1, # Train on all 7616 samples exactly once
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

# Start Training
trainer_stats = trainer.train()

In [ ]:
model.save_pretrained("therapy_teacher_lora")
tokenizer.save_pretrained("therapy_teacher_lora")